# GridNet-HD — pack the 2D cache into one compact tar per zone (one-time, **CPU runtime**)

ทำไม: Colab อ่าน Drive ผ่าน FUSE ได้แค่ ~10–15 MB/s เมื่ออ่านไฟล์เล็กทีละไฟล์ แต่ >100 MB/s กับไฟล์ใหญ่ไฟล์เดียว
notebook นี้อ่าน `GridNet/cache2d/<RUN2D>/<zone>/*.npz` (fp16 ~50 MB/ภาพ หรือ compact) → เขียน **`<zone>.tar`** (compact uint8/int8
~24 MB/ภาพ) + `<zone>.tar.json` ไว้ข้าง ๆ โฟลเดอร์เดิม (ไฟล์เดิมไม่ถูกแตะ) → `colab_train_fusion` cell 5 จะดึง tar แทนการ copy ทีละไฟล์

Runtime: **CPU** (ธรรมดาก็ได้; High-RAM ถ้าเลือกได้) — ไม่ใช้ GPU, ใช้แค่ numpy · 14 zones ≈ 2,760 ภาพ ≈ 140 GB fp16 → ≈ 67 GB tar ·
Drive mount อ่านได้ ~20–25 MB/s รวม (วัดจากโซนแรก ๆ: 111 ภาพ 4.7 นาที) → ทั้งหมดราว **2–2.5 ชม.**
การอ่านผ่าน Drive mount, แต่ **การอัปโหลด tar ใช้ Drive API โดยตรง** (resumable, ทีละ 32 MB) เพราะการเขียนไฟล์หลาย GB ผ่าน mount ทำให้ Drive client
ของ VM โดนฆ่าตอน RAM ตึง (`exit -9` / `Transport endpoint is not connected`) · cell 3 จะขอสิทธิ์ Google account 1 ครั้ง (popup)
**Resume**: รันซ้ำได้ — โซนที่มี tar ครบบน Drive แล้วจะถูกข้าม, ไฟล์ที่แปลงแล้วบนดิสก์ local ของ session เดิมถูกใช้ต่อ ·
ถ้า Drive mount ตาย cell 3 จะ remount แล้วทำโซนนั้นต่อเอง

In [ ]:
#@title 1. Mount Drive + parameters
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/GridNet'
CODE_ZIP = f'{DRIVE}/code/gridnet_code.zip'      #@param {type:"string"}
RUN2D = 'upernet_convnext_t_v2'                  #@param {type:"string"}
ZONES = 't1z6a,t2z5,t3z3,t5a1,t5b2,t5c1,t6z2,t6z4,t1z5b,t1z8,t3z4,t4z1,t5b1,t5b5'   #@param {type:"string"}
THREADS = 4               #@param {type:"integer"}
WORK = '/content/cache2d_work'
CACHE2D_DRIVE = f'{DRIVE}/cache2d/{RUN2D}'
assert os.path.isdir(CACHE2D_DRIVE), CACHE2D_DRIVE
zones = [z.strip() for z in ZONES.split(',') if z.strip()]
have = {z: len([f for f in os.listdir(f'{CACHE2D_DRIVE}/{z}') if f.endswith('.npz')]) if os.path.isdir(f'{CACHE2D_DRIVE}/{z}') else 0 for z in zones}
print({z: n for z, n in have.items()}); assert all(have.values()), f"zones without a 2D cache: {[z for z, n in have.items() if not n]}"


In [ ]:
#@title 2. Code (system Python is enough: numpy only)
import zipfile, os, subprocess, sys
os.makedirs('/content/code', exist_ok=True); zipfile.ZipFile(CODE_ZIP).extractall('/content/code')
r = subprocess.run([sys.executable, '-c', "import sys; sys.path.insert(0,'/content/code'); from gridnet.train.maps2d import write_zone_tar; print('imports ok')"],
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-1500:])
print(subprocess.run(['df', '-h', '/content'], capture_output=True, text=True).stdout)
ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
print(f"RAM {ram_gb:.0f} GB")
if ram_gb < 20: print("standard runtime (12.7 GB): fine - tars are uploaded through the Drive API, not the mount")


In [ ]:
#@title 3. Convert (reads through the mount) + upload each zone tar through the Drive API (resumable; finished zones skipped)
import subprocess, sys, time, os, json
from google.colab import drive, auth
sys.path.insert(0, '/content/code')
from gridnet.train import drive_api
auth.authenticate_user()                                   # once per session (popup) - Drive API scope
svc = drive_api.drive_service()
folder_id = drive_api.resolve_path(svc, ['GridNet', 'cache2d', RUN2D])
print("Drive folder", f"GridNet/cache2d/{RUN2D}", "=", folder_id)
LOCAL_TARS = f'{WORK}/tars'
env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore', MALLOC_ARENA_MAX='2')

def convert(zone):
    cmd = [sys.executable, '/content/code/scripts/compact_cache2d.py', '--cache2d', CACHE2D_DRIVE, '--zones', zone,
           '--work', WORK, '--threads', str(THREADS), '--no-publish', '--pause', '0']
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    try:
        for line in p.stdout: print(line, end='')
        p.wait()
    except KeyboardInterrupt:
        p.terminate(); p.wait(); raise
    return p.returncode

def remount():
    try: drive.flush_and_unmount()
    except Exception as e: print("  unmount:", str(e)[:200])
    time.sleep(15); drive.mount('/content/drive', force_remount=True)

def upload(path, mimetype):
    last = [0]
    def prog(done, total):
        if done - last[0] >= 512 << 20 or done == total:
            last[0] = done; print(f"\r  upload {os.path.basename(path)}: {done / 1e9:.1f}/{total / 1e9:.1f} GB", end='', flush=True)
    r = drive_api.upload_file(svc, path, folder_id, mimetype=mimetype, progress=prog)
    print(f"  ok ({int(r['size']) / 1e9:.2f} GB in {r['seconds'] / 60:.1f} min, {int(r['size']) / 1e6 / max(r['seconds'], 1e-3):.0f} MB/s)")
    return r

t0 = time.time(); done_zones = []
try:
    for z in zones:
        m = drive_api.zone_tar_on_drive(svc, folder_id, z)
        if m is not None:
            print(f"[{z}] already on Drive ({m['n']} images, {m['bytes'] / 1e9:.1f} GB) - skip"); done_zones.append(z); continue
        tar, man = f'{LOCAL_TARS}/{z}.tar', f'{LOCAL_TARS}/{z}.tar.json'
        for attempt in range(1, 8):                        # the mount may die while reading: remount and resume the zone
            if os.path.exists(tar) and os.path.exists(man) and os.path.getsize(tar) == json.load(open(man))['bytes']:
                rc = 0; break
            rc = convert(z)
            if rc == 0: break
            if rc == -9: print("converter killed (out of memory) - lower THREADS or use a High-RAM runtime"); break
            print(f"[{z}] converter exit {rc} (attempt {attempt}) -> remounting Drive", flush=True); remount()
        if rc != 0 or not os.path.exists(tar): print(f"[{z}] FAILED - re-run this cell"); break
        upload(tar, 'application/x-tar'); upload(man, 'application/json')
        os.remove(tar); os.remove(man); done_zones.append(z)
        print(f"[{z}] done  ({(time.time() - t0) / 60:.1f} min elapsed, {len(done_zones)}/{len(zones)} zones)", flush=True)
except KeyboardInterrupt:
    print("stopped; re-run this cell to continue")
print(f"{len(done_zones)}/{len(zones)} zones on Drive after {(time.time() - t0) / 60:.1f} min")


In [ ]:
#@title 4. Check what is on Drive now (through the API, works even if the mount is dead)
import sys
sys.path.insert(0, '/content/code')
from gridnet.train import drive_api
from google.colab import auth
auth.authenticate_user()
svc = drive_api.drive_service()
folder_id = drive_api.resolve_path(svc, ['GridNet', 'cache2d', RUN2D])
total = 0
for z in zones:
    m = drive_api.zone_tar_on_drive(svc, folder_id, z)
    if m is None: print(f"  {z}: NO complete tar"); continue
    total += m['bytes']; print(f"  {z}: {m['n']} images  {m['bytes'] / 1e9:.1f} GB")
print(f"total {total / 1e9:.1f} GB of zone tars in GridNet/cache2d/{RUN2D}")


## หมายเหตุ
- โฟลเดอร์ `<zone>/` เดิม (fp16) ยังอยู่ ใช้เป็นสำรองได้; ถ้าต้องการพื้นที่ค่อยลบทีหลังเมื่อ tar ครบแล้ว (Drive มี 4.8 TB ไม่รีบ)
- export โซนใหม่ในอนาคต: รัน colab_train_upernet cell 8 ตามปกติ แล้วรัน notebook นี้เพิ่มเฉพาะโซนนั้น (`ZONES`)
- จบแล้ว **Disconnect and delete runtime**